# Part 5 — Integrated Deterministic Core

### The skeleton for the production lithium model: six stages, recycling loop, dual feedstock, and a regression harness

Parts 1–4 each isolated one mechanism. **This notebook is the first to run them together**, because
the production model needs them simultaneously and feature interaction — not model size — is what
usually breaks a rewrite.

What is combined here for the first time:

| Mechanism | Introduced in | Why it must be here |
|---|---|---|
| Explicit nodes and arcs | Part 3 | trade and routing are decisions, not parameters |
| Semi-continuous sizing | Part 3 | fewer binaries **and** a finer decision space than integer unit counts |
| Vintage indexing | Part 3 | the only way to get **retirement** without a separate variable |
| Variable-length periods | Part 3 | 80 annual periods is the dominant size driver |
| Shortfall with penalty | Part 3 | avoids infeasibility and yields the shadow prices that rank interventions |
| Six-stage chain + recycling | new | the actual lithium chain, closed loop |
| **Dual feedstock** | new | one facility accepting virgin *and* recycled input |

### The design rule carried forward

Integers decide **which facilities exist and how large**. Everything downstream of that —
throughput, flows, recycling, shortfall — is a pure LP. That separation is what keeps Benders
available (Part 2b) and a KKT-based follower possible (Part 4d/4f). Nothing in this notebook
violates it.

### Why dual feedstock is here

The policy argument for regional processing hubs is that one plant should accept both imported
virgin ore and domestically recycled material. In the current production model recycling output
*substitutes* for processed material and bypasses the processing stage entirely, so recycling and
processing are alternatives rather than complements and a dual-feedstock plant cannot be expressed.
Here `CATH` draws on a single input pool fed by **both** `PROC` and `REC`, which is the minimal
change that makes the recommendation testable.

## Formulation reference

### Sets

| Set | Symbol | Members |
|---|---|---|
| Regions | $r \in \mathcal{R}$ | configurable; `R1, R2` by default |
| Chain stages | $s \in \mathcal{S}$ | MINE → PROC → CATH → CELL → PACK |
| Recycling | REC | consumes retired PACK, emits PROC-grade material |
| Periods | $p \in \mathcal{P}$ | unequal length, spanning the horizon |
| Vintages | $v$ | $-1$ legacy; $0\ldots$ build decided in period $v$ |
| Active | $(s,r,v,p)$ | vintage $v$ at $(s,r)$ operating in $p$ — encodes lead time **and** retirement |

### Parameters

| Parameter | Symbol | Meaning |
|---|---|---|
| Period length | $L_p$ | years in period $p$ |
| Period weight | $\omega_p=\sum_{t\in p}(1+\rho)^{-t}$ | **sum** of annual discount factors |
| Lead time | $\ell_s$ | years from decision to operation |
| Asset life | $\Lambda_s$ | operating years before retirement |
| Yield | $\eta_s$ | output per unit input |
| Recovery | $\theta$ | PROC-grade material per unit retired pack |
| Pack lifetime | $\Lambda^{pack}$ | years from sale to scrap |
| Transport | $\tau_{r_1r_2}$ | per unit, per arc |
| Shortfall penalty | $\pi$ | per unit unmet demand |

### Decision variables

| Variable | Symbol | Domain |
|---|---|---|
| Build | $y_{s,r,v}$ | $\{0,1\}$ |
| Size | $c_{s,r,v}$ | $\ge 0$, semi-continuous |
| Throughput | $x_{s,r,v,p}$ | $\ge 0$ |
| Flow | $f_{s,r_1,r_2,p}$ | $\ge 0$ |
| Shortfall | $u_{r,p}$ | $\ge 0$ |

### Constraints

**Semi-continuous sizing**
$$\underline{c}\,y_{s,r,v} \;\le\; c_{s,r,v} \;\le\; \overline{c}\,y_{s,r,v}$$

**Capacity limits throughput** — the integer/LP interface:
$$x_{s,r,v,p} \le c_{s,r,v}$$

**Node output onto arcs**
$$\sum_v \eta_s\,x_{s,r,v,p} \;=\; \sum_{r_2} f_{s,r,r_2,p}$$

**Node input — dual feedstock at CATH:**
$$\sum_{r_1} f_{\text{PROC},r_1,r,p} \;+\; \sum_{r_1} f_{\text{REC},r_1,r,p}
\;=\; \sum_v x_{\text{CATH},r,v,p}$$

**Recycling availability** — scrap is what was sold one battery-lifetime ago:
$$\sum_v x_{\text{REC},r,v,p} \;\le\; \theta \sum_{r_1} f_{\text{PACK},r_1,r,\,p-\Lambda^{pack}}$$

**Demand**
$$\sum_{r_1} f_{\text{PACK},r_1,r,p} + u_{r,p} \;\ge\; D_{r,p}$$

### Objective

$$\min \sum \mu_{s,v}\big(F_s y + U_s c\big) \;+\; \sum_p \omega_p\Big(\sum o_s x + \sum \tau f + \pi u\Big)$$

$\omega_p$ — not 1 — on every operating term. A 9-year period weighted as one year understates its
operating cost by ~9×.

## 1. Setup

In [ ]:
import math, itertools
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# WLS licence if present in the folder; otherwise the restricted pip licence.
import os
LIC = None
for cand in ("gurobi.lic", os.path.join("..", "gurobi.lic")):
    if os.path.exists(cand):
        LIC = os.path.abspath(cand); break
if LIC:
    os.environ["GRB_LICENSE_FILE"] = LIC
    print("licence:", LIC)
else:
    print("licence: default (restricted)")

pd.set_option("display.width", 120)
print("gurobi", gp.gurobi.version())


## 2. Configuration

Everything that changes between the demo instance, the collapse test, and the calibrated run lives
in one dict. `build()` reads only from here — that is what makes the regression harness in §9
possible.

In [ ]:
BASE = dict(
    regions      = ["R1", "R2"],
    chain        = ["MINE", "PROC", "CATH", "CELL", "PACK"],
    # period structure: (n_periods, years_each) blocks -> variable length
    period_plan  = [(6, 1), (4, 3), (3, 5)],      # 6x1 + 4x3 + 3x5 = 33 years, 13 periods
    rho          = 0.05,
    lead         = {"MINE": 3, "PROC": 2, "CATH": 1, "CELL": 1, "PACK": 1, "REC": 1},
    life         = {"MINE": 20, "PROC": 20, "CATH": 15, "CELL": 12, "PACK": 12, "REC": 15},
    yield_       = {"MINE": 0.95, "PROC": 0.90, "CATH": 0.92, "CELL": 0.94, "PACK": 0.98, "REC": 0.85},
    fixed_cost   = {"MINE": 300., "PROC": 260., "CATH": 180., "CELL": 220., "PACK": 90., "REC": 140.},
    unit_cost    = {"MINE": 2.4, "PROC": 3.1, "CATH": 2.2, "CELL": 3.6, "PACK": 1.1, "REC": 1.9},
    op_cost      = {"MINE": 0.9, "PROC": 1.4, "CATH": 1.1, "CELL": 1.7, "PACK": 0.5, "REC": 0.8},
    cap_min      = 8.0,
    cap_max      = 60.0,
    tau_intra    = 0.3,
    tau_inter    = 1.6,
    penalty      = 40.0,
    pack_life    = 10,          # years from sale to scrap
    recovery     = 0.55,        # PROC-grade material recovered per retired pack unit
    demand0      = {"R1": 30.0, "R2": 18.0},
    demand_growth= 0.045,
    legacy_cap   = {("MINE","R1"): 12.0, ("PROC","R1"): 10.0, ("CATH","R1"): 8.0,
                    ("CELL","R1"): 8.0,  ("PACK","R1"): 8.0,
                    ("MINE","R2"): 6.0,  ("PROC","R2"): 5.0},
    mipgap       = 0.005,
    allow_dual_feedstock = True,
)

def periods_from_plan(plan):
    lens, t0 = [], 0
    for n, L in plan:
        for _ in range(n):
            lens.append(L)
    starts = []
    for L in lens:
        starts.append(t0); t0 += L
    return lens, starts, t0

LENS, STARTS, HORIZON = periods_from_plan(BASE["period_plan"])
print(f"{len(LENS)} periods, {HORIZON} years:", LENS)


## 3. The model

Written inline in the Part 3 idiom so each block reads close to its algebraic statement. The one
structural difference from Part 3 is the **input pool** at CATH, which is what makes dual feedstock
expressible.

In [ ]:
def build(cfg, verbose=False):
    R      = cfg["regions"]
    CHAIN  = cfg["chain"]
    STAGES = CHAIN + ["REC"]
    rho    = cfg["rho"]

    lens, starts, H = periods_from_plan(cfg["period_plan"])
    P = list(range(len(lens)))

    # period weight = SUM of annual discount factors inside the period
    omega = {p: sum((1+rho)**-(starts[p]+k) for k in range(lens[p])) for p in P}

    # ---- vintages: -1 legacy, else decided in period v -------------------
    VIN = [-1] + P

    def online(s, v, p):
        """is vintage v of stage s operating in period p?  encodes lead time AND retirement"""
        if v == -1:
            return True
        t_ready = starts[v] + cfg["lead"][s]
        t_end   = t_ready + cfg["life"][s]
        return (starts[p] >= t_ready) and (starts[p] < t_end)

    BUILD  = [(s, r, v) for s in STAGES for r in R for v in P]
    ACTIVE = [(s, r, v, p) for s in STAGES for r in R for v in VIN for p in P
              if online(s, v, p) and (v != -1 or (s, r) in cfg["legacy_cap"])]

    ARCS = [(s, r1, r2) for s in STAGES for r1 in R for r2 in R]

    # capex present-value multiplier: CRF x discount factors over operating years in horizon
    def mu(s, v):
        life = cfg["life"][s]
        crf  = rho*(1+rho)**life / ((1+rho)**life - 1)
        t0   = starts[v] + cfg["lead"][s]
        yrs  = [t for t in range(t0, min(t0+life, H))]
        return crf * sum((1+rho)**-t for t in yrs) if yrs else 0.0

    DEM = {(r, p): cfg["demand0"][r]*(1+cfg["demand_growth"])**starts[p] for r in R for p in P}

    m = gp.Model("part5_core")
    m.Params.OutputFlag = 1 if verbose else 0
    m.Params.MIPGap = cfg["mipgap"]

    y = m.addVars(BUILD,  vtype=GRB.BINARY, name="y")
    c = m.addVars(BUILD,  lb=0.0, ub=cfg["cap_max"], name="c")
    x = m.addVars(ACTIVE, lb=0.0, name="x")
    f = m.addVars(ARCS, P, lb=0.0, name="f")
    u = m.addVars(R, P, lb=0.0, name="u")

    # ---- semi-continuous sizing -----------------------------------------
    m.addConstrs((c[s,r,v] <= cfg["cap_max"]*y[s,r,v] for (s,r,v) in BUILD), name="size_hi")
    m.addConstrs((c[s,r,v] >= cfg["cap_min"]*y[s,r,v] for (s,r,v) in BUILD), name="size_lo")

    # ---- capacity limits throughput (integer -> LP interface) -----------
    m.addConstrs((x[s,r,v,p] <= (cfg["legacy_cap"][(s,r)] if v == -1 else c[s,r,v])
                  for (s,r,v,p) in ACTIVE), name="cap")

    def thr(s, r, p):
        return gp.quicksum(x[s,r,v,p] for v in VIN if (s,r,v,p) in x)

    # ---- node output leaves on arcs -------------------------------------
    m.addConstrs((cfg["yield_"][s]*thr(s,r,p) == f.sum(s, r, "*", p)
                  for s in STAGES for r in R for p in P), name="out")

    # ---- node input ------------------------------------------------------
    for i, s in enumerate(CHAIN):
        if i == 0:
            continue                       # MINE draws on reserves, not an arc
        prev = CHAIN[i-1]
        for r in R:
            for p in P:
                inflow = f.sum(prev, "*", r, p)
                if s == "CATH" and cfg["allow_dual_feedstock"]:
                    inflow = inflow + f.sum("REC", "*", r, p)   # <-- dual feedstock
                m.addConstr(inflow == thr(s, r, p), name=f"in_{s}_{r}_{p}")

    # if dual feedstock is switched off, recycled material has nowhere to go
    if not cfg["allow_dual_feedstock"]:
        m.addConstrs((f.sum("REC", "*", r, p) == 0 for r in R for p in P), name="rec_sink")

    # ---- recycling availability: scrap = packs sold one lifetime ago -----
    def pack_period(p):
        """index of the period containing (start[p] - pack_life), or None"""
        t = starts[p] - cfg["pack_life"]
        if t < 0:
            return None
        for q in P:
            if starts[q] <= t < starts[q] + lens[q]:
                return q
        return None

    for r in R:
        for p in P:
            q = pack_period(p)
            if q is None:
                m.addConstr(thr("REC", r, p) == 0, name=f"rec0_{r}_{p}")
            else:
                m.addConstr(thr("REC", r, p) <= cfg["recovery"]*f.sum("PACK", "*", r, q),
                            name=f"rec_{r}_{p}")

    # ---- demand ----------------------------------------------------------
    m.addConstrs((f.sum("PACK", "*", r, p) + u[r,p] >= DEM[r,p] for r in R for p in P),
                 name="dem")

    # ---- objective -------------------------------------------------------
    capex = gp.quicksum(mu(s,v)*(cfg["fixed_cost"][s]*y[s,r,v] + cfg["unit_cost"][s]*c[s,r,v])
                        for (s,r,v) in BUILD)
    opex  = gp.quicksum(omega[p]*cfg["op_cost"][s]*x[s,r,v,p] for (s,r,v,p) in ACTIVE)
    trans = gp.quicksum(omega[p]*(cfg["tau_intra"] if r1 == r2 else cfg["tau_inter"])*f[s,r1,r2,p]
                        for (s,r1,r2) in ARCS for p in P)
    short = gp.quicksum(omega[p]*cfg["penalty"]*u[r,p] for r in R for p in P)
    m.setObjective(capex + opex + trans + short, GRB.MINIMIZE)

    m.update()          # required before .relax() - otherwise it copies an empty model
    m._sets = dict(R=R, STAGES=STAGES, CHAIN=CHAIN, P=P, VIN=VIN, BUILD=BUILD,
                   ACTIVE=ACTIVE, ARCS=ARCS, omega=omega, DEM=DEM, starts=starts, lens=lens)
    m._vars = dict(y=y, c=c, x=x, f=f, u=u)
    return m

## 4. Solve the base instance

In [ ]:
m = build(BASE)
m.optimize()
print("status", m.Status, "| obj %.2f" % m.ObjVal, "| gap %.4f%%" % (100*m.MIPGap))
print("vars %d  constrs %d  binaries %d" % (m.NumVars, m.NumConstrs, m.NumBinVars))


## 5. What got built

Vintage indexing means retirement is implicit: a facility simply stops appearing in `ACTIVE` once
`start[p] >= ready + life`. There is no retirement variable and no decommissioning constraint.

In [ ]:
S = m._sets; V = m._vars
rows = []
for (s, r, v) in S["BUILD"]:
    if V["y"][s, r, v].X > 0.5:
        rows.append(dict(stage=s, region=r, vintage=v,
                         year=S["starts"][v], size=round(V["c"][s, r, v].X, 2)))
built = pd.DataFrame(rows).sort_values(["year", "stage", "region"])
print(built.to_string(index=False) if len(built) else "nothing built")
print("\ntotal capacity added: %.1f" % built["size"].sum() if len(built) else "")

In [ ]:
# unmet demand and recycling contribution over time
rec_share = []
for p in S["P"]:
    packs = sum(V["f"][("PACK", r1, r2)][p].X if False else V["f"]["PACK", r1, r2, p].X
                for r1 in S["R"] for r2 in S["R"])
    recf  = sum(V["f"]["REC", r1, r2, p].X for r1 in S["R"] for r2 in S["R"])
    proc  = sum(V["f"]["PROC", r1, r2, p].X for r1 in S["R"] for r2 in S["R"])
    short = sum(V["u"][r, p].X for r in S["R"])
    rec_share.append(dict(period=p, year=S["starts"][p], packs=round(packs, 2),
                          proc=round(proc, 2), rec=round(recf, 2),
                          rec_pct=round(100*recf/(proc+recf), 1) if (proc+recf) > 1e-6 else 0.0,
                          unmet=round(short, 2)))
print(pd.DataFrame(rec_share).to_string(index=False))


## 6. Does dual feedstock matter?

Switch it off and recycled material has nowhere to go. The difference is the value of allowing one
facility to accept both streams — the quantity the policy brief asserts but the current production
model cannot express.

In [ ]:
cfg_off = dict(BASE); cfg_off["allow_dual_feedstock"] = False
m_off = build(cfg_off); m_off.optimize()

print("dual feedstock ON : %.2f" % m.ObjVal)
print("dual feedstock OFF: %.2f" % m_off.ObjVal)
delta = m_off.ObjVal - m.ObjVal
print("value of dual feedstock: %.2f  (%.2f%%)" % (delta, 100*delta/m_off.ObjVal))


## 7. Diagnostic — is the semi-continuous bound doing anything?

If every built facility sits exactly at `cap_max`, the sizing decision is degenerate and you have
paid for continuous variables that behave like a fixed unit size. If sizes spread across the
interval, the semi-continuous formulation is earning its place.

In [ ]:
if len(built):
    at_max = (built["size"] >= BASE["cap_max"] - 1e-6).sum()
    at_min = (built["size"] <= BASE["cap_min"] + 1e-6).sum()
    print("facilities built : %d" % len(built))
    print("at cap_max       : %d" % at_max)
    print("at cap_min       : %d" % at_min)
    print("strictly interior: %d" % (len(built) - at_max - at_min))
    print("\nsize distribution:\n", built["size"].describe().round(2).to_string())
    if len(built) - at_max - at_min == 0:
        print("\n[!] all sizes at a bound - semi-continuous sizing is not buying anything here")


## 8. Diagnostic — period weighting

The single most common bug in a variable-period model is weighting a 5-year period as one year.
This check confirms $\sum_p \omega_p$ equals the discounted value of a unit annual stream over the
whole horizon, computed independently.

In [ ]:
rho = BASE["rho"]
direct = sum((1+rho)**-t for t in range(HORIZON))
summed = sum(m._sets["omega"].values())
print("sum of period weights : %.6f" % summed)
print("independent annual sum: %.6f" % direct)
assert abs(summed - direct) < 1e-9, "period weights do not tile the horizon"
print("OK - weights tile the horizon exactly")


## 9. The regression harness

Two layers, and they answer different questions.

**Layer 1 — structural invariant (runs here).** Take the multi-region model, make every region
identical, and set inter-region transport to the intra-region cost. Trade is then free and regions
are interchangeable, so the optimum must equal a **single-region model carrying the summed demand**.

Run it on the **LP relaxation**, not the MILP. The invariant is exact only when capacity is
continuous: with semi-continuous sizing the two configurations have genuinely different feasible
sets, because two facilities of size `cap_min` are reachable in the two-region layout and a single
facility of `2 × cap_min` is a different point. On this instance that lumpiness alone produces a
**0.6% MILP discrepancy** — large enough to swamp the signal a regression test is supposed to carry.
Relaxing the binaries removes it and any residual difference is then a plumbing error in the arc or
balance logic.

The MILP gap is still worth printing as a diagnostic: it measures how much the integer layer is
distorting, which is exactly what you want to watch when tuning `cap_min`.

**Layer 2 — data regression (hook provided).** Point `CALIBRATED` at the 2020–2100 dataset and
assert the one-region objective reproduces the published \$9.51T. That is the test that converts
this from *a nicer model* into *a validated replacement*, and it should be run on every commit once
the data is wired in.

In [ ]:
def collapse_test(cfg, tol=1e-4):
    # multi-region with free trade and identical regions
    # must equal one region carrying the summed demand
    multi = dict(cfg)
    multi["regions"]   = ["A", "B"]
    multi["tau_inter"] = cfg["tau_intra"]          # trade is free -> geography is irrelevant
    multi["demand0"]   = {"A": 12.0, "B": 12.0}
    multi["legacy_cap"] = {(s, r): 6.0 for s in cfg["chain"] for r in ["A", "B"]}
    multi["mipgap"]    = 0.0

    single = dict(multi)
    single["regions"]  = ["A"]
    single["demand0"]  = {"A": 24.0}
    single["legacy_cap"] = {(s, "A"): 12.0 for s in cfg["chain"]}

    a = build(multi);  b = build(single)

    # --- the actual test: LP relaxation, where the invariant is exact -----
    ra, rb = a.relax(), b.relax()
    ra.Params.OutputFlag = 0; rb.Params.OutputFlag = 0
    ra.optimize(); rb.optimize()
    # guard: a silently empty relaxation solves to 0 and would pass any test
    assert ra.NumConstrs > 0 and rb.NumConstrs > 0, "relaxation is empty - missing m.update()"
    assert ra.ObjVal > 1.0, "relaxed objective is ~0; the model is not being copied"
    rel_lp = abs(ra.ObjVal - rb.ObjVal) / max(1.0, abs(rb.ObjVal))

    # --- diagnostic only: how much does the integer layer distort? --------
    a.optimize(); b.optimize()
    rel_ip = abs(a.ObjVal - b.ObjVal) / max(1.0, abs(b.ObjVal))

    print("LP relaxation   multi %.6f | single %.6f | rel %.3e   <-- the test" %
          (ra.ObjVal, rb.ObjVal, rel_lp))
    print("MILP            multi %.4f | single %.4f | rel %.3e   (lumpiness)" %
          (a.ObjVal, b.ObjVal, rel_ip))
    return rel_lp, rel_ip

rel_lp, rel_ip = collapse_test(BASE)
assert rel_lp < 1e-6, f"collapse invariant violated in the LP: {rel_lp:.3e}"
print("\nPASS - arc and balance logic collapses exactly under relaxation")
print("integer lumpiness contributes %.2f%% on this instance" % (100*rel_ip))

In [ ]:
# ---- Layer 2: hook for the calibrated dataset -------------------------------
CALIBRATED = None      # set to a cfg dict built from input_csvs/ when available
PUBLISHED_OBJECTIVE = 9.51e6     # Jones (2024) Energies 17:2685, cost objective

def data_regression(cfg, target=PUBLISHED_OBJECTIVE, tol=0.02):
    if cfg is None:
        print("SKIPPED - CALIBRATED not set. Wire input_csvs/ into a cfg dict to enable.")
        return None
    one = dict(cfg); one["regions"] = ["World"]
    mm = build(one); mm.optimize()
    rel = abs(mm.ObjVal - target)/target
    print("one-region objective : %.4g" % mm.ObjVal)
    print("published target     : %.4g" % target)
    print("relative difference  : %.3f%%" % (100*rel))
    assert rel < tol, "REGRESSION FAILED - the refactor changed the answer"
    print("PASS")
    return mm.ObjVal

data_regression(CALIBRATED)

## 10. Where this goes

This notebook is the skeleton, not the model. What layers on top, in order:

1. **CO₂ as a second accounting stream** — mirrors the cost terms exactly; no structural change.
2. **Stochastics** — the first stage is already `y` and `c`; recourse is already a pure LP.
   That is the shape Part 2b (L-shaped) and Part 2c (CVaR) require.
3. **Regions beyond two**, and a trade-cost matrix rather than an intra/inter switch.
4. **Interdiction** — Part 4f attacks the arcs defined here.

Two things deliberately **not** here: learning curves (they fight SOS2 once the driver becomes
production — Part 3b) and any game structure. Both belong on top of a core that is known to solve.

### Known limitations of this instance

- Reserve limits are absent; MINE is bounded only by capacity.
- One commodity per stage. Multi-mineral needs a recipe matrix, not another index.
- Retirement is deterministic by vintage age; no early decommissioning decision.
- The recycling lag maps to the period containing `start[p] - pack_life`, so it is exact only at
  period boundaries. With the 1/3/5-year blocks used here that is accurate to within one period.